In [1]:
!pip install -q -U cohere

In [2]:
from google.colab import userdata
import cohere

COHERE_API_KEY = userdata.get("COHERE_API_KEY")
client = cohere.ClientV2(api_key=COHERE_API_KEY)

response = client.chat(
    model="command-r-08-2024",
    messages=[
        {
            "role": "user",
            "content": "Classify this ESG message as JSON only: "
                       "There is a water leak in Building C."
        }
    ]
)

print(response.message.content[0].text)

{
  "ESG_message": "There is a water leak in Building C.",
  "classification": "Environmental"
}


In [3]:

import json
import pandas as pd
import cohere
from google.colab import userdata
from IPython.display import display

# 1. Connect to Cohere using your private Colab Secret
COHERE_API_KEY = userdata.get("COHERE_API_KEY")

if not COHERE_API_KEY:
    raise ValueError(
        "COHERE_API_KEY not found. Check Colab Secrets and notebook access."
    )

client = cohere.ClientV2(api_key=COHERE_API_KEY)

# 2. Revised ESG triage prompt
SYSTEM_PROMPT = """
You are an ESG operational message triage assistant.
Classify the employee message using only the evidence provided.
Do not invent facts or claim that an action has been completed.

Return a JSON object containing exactly these fields:
issue_category, urgency, sentiment, followup_required,
recommended_team, escalation_reason, data_sensitivity_risk,
brief_summary.

Allowed issue_category values:
WATER, WASTE_RECYCLING, ENERGY, SUPPLIER_SUSTAINABILITY,
ACCESSIBILITY, GOVERNANCE, OTHER.

Allowed urgency values: LOW, MEDIUM, HIGH, CRITICAL.
LOW: minor issue with no apparent immediate impact.
MEDIUM: recurring or operational issue requiring follow-up.
HIGH: significant ongoing disruption, blocked accessibility,
or material compliance concern.
CRITICAL: immediate threat to life or safety, or a severe
active hazard supported by the message.

Allowed sentiment values: POSITIVE, NEUTRAL, NEGATIVE.
followup_required must be Y or N.
Allowed recommended_team values:
FACILITIES, SUSTAINABILITY, PROCUREMENT, ACCESSIBILITY,
GOVERNANCE_COMPLIANCE, HUMAN_REVIEW.
data_sensitivity_risk must be LOW, MEDIUM, or HIGH.

Treat supplier allegations as unverified, not proven violations.
Assess privacy risk using information actually present.
Explain uncertainty where appropriate.
Do not assume an emergency if the message does not support it.
Recommend the relevant emergency pathway when immediate danger
is indicated.
Return JSON only.
"""

# 3. Three realistic sample messages
sample_messages = [
    {
        "id": "ESG-01",
        "message": (
            "There is a water leak in Building C that has been "
            "running all morning."
        )
    },
    {
        "id": "ESG-02",
        "message": (
            "The accessible entrance near the main building has "
            "been blocked for two days."
        )
    },
    {
        "id": "ESG-03",
        "message": (
            "One of our suppliers may not meet our sustainability "
            "policy. Please investigate."
        )
    }
]

# 4. Run the LLM and validate its JSON output
results = []

for item in sample_messages:
    print(f"\nProcessing {item['id']} ...")

    try:
        response = client.chat(
            # Use the model that worked in your connection test
            model="command-r-08-2024",
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": (
                        "Classify this ESG message and return JSON only:\n"
                        + item["message"]
                    )
                }
            ],
            response_format={"type": "json_object"},
            max_tokens=400
        )

        output_text = response.message.content[0].text
        parsed = json.loads(output_text)

        expected_fields = [
            "issue_category",
            "urgency",
            "sentiment",
            "followup_required",
            "recommended_team",
            "escalation_reason",
            "data_sensitivity_risk",
            "brief_summary"
        ]

        # Check required fields
        missing = [key for key in expected_fields if key not in parsed]
        if missing:
            raise ValueError(f"Missing JSON fields: {missing}")

        # Check permitted values
        assert parsed["issue_category"] in {
            "WATER", "WASTE_RECYCLING", "ENERGY",
            "SUPPLIER_SUSTAINABILITY", "ACCESSIBILITY",
            "GOVERNANCE", "OTHER"
        }
        assert parsed["urgency"] in {
            "LOW", "MEDIUM", "HIGH", "CRITICAL"
        }
        assert parsed["sentiment"] in {
            "POSITIVE", "NEUTRAL", "NEGATIVE"
        }
        assert parsed["followup_required"] in {"Y", "N"}
        assert parsed["recommended_team"] in {
            "FACILITIES", "SUSTAINABILITY", "PROCUREMENT",
            "ACCESSIBILITY", "GOVERNANCE_COMPLIANCE",
            "HUMAN_REVIEW"
        }
        assert parsed["data_sensitivity_risk"] in {
            "LOW", "MEDIUM", "HIGH"
        }

        results.append({
            "id": item["id"],
            "message": item["message"],
            "llm_result": parsed,
            "status": "PASS"
        })

        print(json.dumps(parsed, indent=2, ensure_ascii=False))
        print("JSON and field validation: PASS")

    except Exception as error:
        results.append({
            "id": item["id"],
            "message": item["message"],
            "llm_result": None,
            "status": f"ERROR: {error}"
        })
        print("Error:", error)

# 5. Display a compact summary table
summary_rows = []

for result in results:
    row = {
        "ID": result["id"],
        "Message": result["message"],
        "Validation": result["status"]
    }

    if result["llm_result"]:
        row.update(result["llm_result"])

    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows)
display(results_df)

# 6. Save actual results for your assignment
with open("esg_llm_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

results_df.to_csv("esg_llm_results.csv", index=False)

print("\nSaved: esg_llm_results.json")
print("Saved: esg_llm_results.csv")



Processing ESG-01 ...
{
  "issue_category": "WATER",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "FACILITIES",
  "escalation_reason": "Ongoing water leak with potential for significant damage and disruption.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Water leak in Building C requires immediate attention."
}
JSON and field validation: PASS

Processing ESG-02 ...
{
  "issue_category": "ACCESSIBILITY",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "ACCESSIBILITY",
  "escalation_reason": "Blocking the accessible entrance for an extended period can significantly impact individuals with accessibility needs and may violate compliance standards.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "The accessible entrance near the main building has been obstructed for two days, potentially hindering access for individuals with disabilities."
}
JSON and field validation: PAS

,ID,Message,Validation,issue_category,urgency,sentiment,followup_required,recommended_team,escalation_reason,data_sensitivity_risk,brief_summary
0,ESG-01,There is a water leak in Building C that has b...,PASS,WATER,HIGH,NEGATIVE,Y,FACILITIES,Ongoing water leak with potential for signific...,LOW,Water leak in Building C requires immediate at...
1,ESG-02,The accessible entrance near the main building...,PASS,ACCESSIBILITY,HIGH,NEGATIVE,Y,ACCESSIBILITY,Blocking the accessible entrance for an extend...,LOW,The accessible entrance near the main building...
2,ESG-03,One of our suppliers may not meet our sustaina...,PASS,SUPPLIER_SUSTAINABILITY,MEDIUM,NEGATIVE,Y,PROCUREMENT,Potential non-compliance with sustainability s...,LOW,Supplier sustainability concern requires inves...



Saved: esg_llm_results.json
Saved: esg_llm_results.csv


In [4]:

def rule_based_triage(message):
    text = message.lower()

    if "accessible entrance" in text or "accessibility" in text:
        category = "ACCESSIBILITY"
        urgency = "HIGH"
        team = "ACCESSIBILITY"
    elif "water leak" in text or "leak" in text:
        category = "WATER"
        urgency = "HIGH" if "all morning" in text else "MEDIUM"
        team = "FACILITIES"
    elif "supplier" in text or "sustainability policy" in text:
        category = "SUPPLIER_SUSTAINABILITY"
        urgency = "MEDIUM"
        team = "PROCUREMENT"
    elif "recycling" in text or "bins" in text:
        category = "WASTE_RECYCLING"
        urgency = "MEDIUM"
        team = "SUSTAINABILITY"
    elif "air conditioning" in text or "overnight" in text:
        category = "ENERGY"
        urgency = "LOW"
        team = "FACILITIES"
    else:
        category = "OTHER"
        urgency = "MEDIUM"
        team = "HUMAN_REVIEW"

    sentiment = (
        "NEGATIVE"
        if any(word in text for word in [
            "blocked", "leak", "again", "may not", "problem"
        ])
        else "NEUTRAL"
    )

    return {
        "issue_category": category,
        "urgency": urgency,
        "sentiment": sentiment,
        "followup_required": "Y",
        "recommended_team": team,
        "escalation_reason": (
            "Review according to the reported issue and urgency."
        ),
        "data_sensitivity_risk": (
            "MEDIUM" if "supplier" in text else "LOW"
        ),
        "brief_summary": message
    }


baseline_rows = []

for item in sample_messages:
    baseline_rows.append({
        "ID": item["id"],
        "Message": item["message"],
        **rule_based_triage(item["message"])
    })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

baseline_df.to_csv("esg_rule_based_results.csv", index=False)
print("Saved: esg_rule_based_results.csv")


,ID,Message,issue_category,urgency,sentiment,followup_required,recommended_team,escalation_reason,data_sensitivity_risk,brief_summary
0,ESG-01,There is a water leak in Building C that has b...,WATER,HIGH,NEGATIVE,Y,FACILITIES,Review according to the reported issue and urg...,LOW,There is a water leak in Building C that has b...
1,ESG-02,The accessible entrance near the main building...,ACCESSIBILITY,HIGH,NEGATIVE,Y,ACCESSIBILITY,Review according to the reported issue and urg...,LOW,The accessible entrance near the main building...
2,ESG-03,One of our suppliers may not meet our sustaina...,SUPPLIER_SUSTAINABILITY,MEDIUM,NEGATIVE,Y,PROCUREMENT,Review according to the reported issue and urg...,MEDIUM,One of our suppliers may not meet our sustaina...


Saved: esg_rule_based_results.csv
